In [1]:
import os
import tifffile
import nibabel as nib
import numpy as np
from nnunetv2.dataset_conversion.generate_dataset_json import generate_dataset_json

In [2]:
SEMANTIC_COLORS = np.array([
    [  0,   0,   0,   0],    # 0: Background
    [  0,  40, 255, 255],    # 1: Cell
    [  0, 212, 255, 255],    # 2: Mitochondrion
    [124, 255, 121, 255],    # 3: Alpha granule
    [255, 229,   0, 255],    # 4: Canalicular vessel
    [255,  70,   0, 255],    # 5: Dense granule body
    [127,   0,  127, 255],    # 6: Dense granule core (Fixed purple logic)
], dtype=np.uint8)

platelet_em_index_to_class = {
    2: 'mitochondrion',
    3: 'alpha granule',
    4: 'canalicular vessel',
    5: 'dense granule',
    6: 'dense granule core'
}

DATA_DIR = '../../data/platelet-em' # directory of images and labels-semantic, TaskXX will be generated into here


In [3]:
def get_binary_mask(rgba_stack, class_idx):
    """Extracts a binary mask for a specific class based on the RGBA color."""
    target_color = SEMANTIC_COLORS[class_idx]
    # Match all 4 channels (R, G, B, A)
    mask = np.all(rgba_stack == target_color, axis=-1)
    return mask.astype(np.uint8)

In [ ]:
def create_msd_task_overlapping(target_idx, task_id, class_name, img_path, lbl_path, window_size=400, stride=40):
    task_name = f"Dataset{task_id}_Platelet{class_name.replace(' ', '').capitalize()}"
    target_dir = os.path.join(
        DATA_DIR, f'Task{target_idx}_{class_name.replace(' ', '')}')

    os.makedirs(os.path.join(target_dir, "imagesTr"), exist_ok=True)
    os.makedirs(os.path.join(target_dir, "labelsTr"), exist_ok=True)

    # Load data (Z, H, W)
    img_data = tifffile.imread(img_path)
    rgba_labels = tifffile.imread(lbl_path)

    # Binarize labels
    binary_label = get_binary_mask(rgba_labels, target_idx)

    Z, H, W = img_data.shape
    spacing = (10.0, 10.0, 40.0)
    affine = np.eye(4)
    count = 0

    # Sliding window with stride of 30
    # Range ends at (Dimension - window_size + 1) to ensure we don't go out of bounds
    for y in range(0, H - window_size + 1, stride):
        for x in range(0, W - window_size + 1, stride):

            # Extract windows (keeping original depth Z)
            img_window = img_data[:, y: y + window_size, x: x + window_size]
            lbl_window = binary_label[:, y: y +
                                      window_size, x: x + window_size]

            # Transpose to NIfTI orientation (X, Y, Z)
            # (H, W, Z) conversion for NIfTI format
            img_nii_data = img_window.transpose(1, 2, 0).astype(np.float32)
            lbl_nii_data = lbl_window.transpose(1, 2, 0).astype(np.uint8)

            # Create a unique filename based on the top-left coordinate (y, x)
            file_id = f"platelet_{target_idx}_y{y}_x{x}"

            # Save NIfTI Image
            img_nii = nib.Nifti1Image(img_nii_data, affine)
            img_nii.header.set_zooms(spacing)
            nib.save(img_nii, os.path.join(
                target_dir, "imagesTr", f"{file_id}.nii.gz"))

            # Save NIfTI Label
            lbl_nii = nib.Nifti1Image(lbl_nii_data, affine)
            lbl_nii.header.set_zooms(spacing)
            nib.save(lbl_nii, os.path.join(
                target_dir, "labelsTr", f"{file_id}.nii.gz"))

            count += 1

    # MSD-style JSON configuration
    generate_dataset_json(
        target_dir,
        channel_names={0: "SEM"},
        labels={"background": 0, class_name: 1},
        num_training_cases=count,
        file_ending=".nii.gz",
        dataset_name=class_name,
        converted_by='Sebastian Ossner',
        license='GNU General Public License v3.0',
        description=f'Dataset generated from the platelet nifti file. Window size={window_size}, stride={stride}'
    )

    print(
        f"Successfully created {count} overlapping windows for Task: {task_name}")

In [ ]:
for class_int, class_name in platelet_em_index_to_class.items():
    create_msd_task_overlapping(
        class_int,
        500 + class_int,
        class_name,
        f"{DATA_DIR}/images/50-images.tif", f"{DATA_DIR}/labels-semantic/50-semantic.tif"
    )

Successfully created 121 overlapping windows for Task: Dataset502_PlateletMitochondrion
Successfully created 121 overlapping windows for Task: Dataset503_PlateletAlphagranule
Successfully created 121 overlapping windows for Task: Dataset504_PlateletCanalicularvessel
Successfully created 121 overlapping windows for Task: Dataset505_PlateletDensegranule
Successfully created 121 overlapping windows for Task: Dataset506_PlateletDensegranulecore
